In [ ]:
"""
This script is to extract the imputed region matrix from pycistopic

authors: Roy Oelen
"""


In [1]:
import os
import pycisTopic
import pandas as pd
import pickle
from scipy import sparse, io
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-10-04 08:31:29,712	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# location of the existing cistopic objects
cistopic_objects_loc='/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/monocytes_all_cells/cistopic_objects/'
# the object to use
cistopic_object_loc=''.join([cistopic_objects_loc, 'cistopic_obj_20_topics_model.pkl'])
# read the object
with open(cistopic_object_loc, 'rb') as f:
    cistopic_object = pickle.load(f)


In [ ]:
# write the csr to an mtx file
mtx_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix.mtx'])
io.mmwrite(mtx_loc, cistopic_object.fragment_matrix)
# we will compress it post-hoc with bgzip /scratch/hb-functionalgenomics/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/matrix.mtx

In [ ]:
# and write the cell names
cell_names = pd.DataFrame(data = {'barcode' : cistopic_object.cell_names})
# to a file
cell_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'barcodes.tsv.gz'])
cell_names.to_csv(cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')

In [ ]:
# and write the region names
feature_names = pd.DataFrame(data = {'barcode' : cistopic_object.region_names})
# to a file
feature_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features.tsv.gz'])
feature_names.to_csv(feature_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')

In [ ]:
# with finally the metadata as well
metadata = cistopic_object.cell_data
# to a file
metadata_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'metadata.tsv.gz'])
metadata.to_csv(metadata_loc, sep = '\t', header = True, index = False, compression = 'gzip')

In [ ]:
# we will also export the region metadata (though we will probably not use it)
regiondata = cistopic_object.region_data
# to a file
regiondata_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'regiondata.tsv.gz'])
regiondata.to_csv(regiondata_loc, sep = '\t', header = True, index = False, compression = 'gzip')

In [ ]:
# impute accessibility
imputed_acc_obj = impute_accessibility(
    cistopic_object,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)
del cistopic_object

In [ ]:
# write the output again, but now for the imputed data
mtx_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix.mtx'])
#io.mmwrite(mtx_loc, sparse.csr_matrix(imputed_acc_obj.mtx))
cell_names = pd.DataFrame(data = {'barcode' : imputed_acc_obj.cell_names})
cell_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'barcodes.tsv.gz'])
cell_names.to_csv(cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
feature_names = pd.DataFrame(data = {'barcode' : imputed_acc_obj.feature_names})
feature_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features.tsv.gz'])
feature_names.to_csv(feature_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')

In [ ]:
# get the number of features
feature_length = len(feature_names)
# set a chunk size
feature_chunk_size = 20000
# start at 0
chunk_pos = 0
# and the number of the index
while chunk_pos < feature_length:
    # we need to set the right of the window
    chunk_pos_right = chunk_pos + feature_chunk_size
    # however if the window on the right exceeds the size, we need to just to use the size
    if chunk_pos_right > feature_length:
        chunk_pos_right = feature_length
    # get the feature names for this chunk
    feature_names_chunk = feature_names[chunk_pos : chunk_pos_right]
    # and the output name
    feature_names_chunk_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features_', str(chunk_pos), '_', str(chunk_pos_right), '.tsv.gz'])
    # get the counts for this chunk
    mtx_chunk = sparse.csr_matrix(imputed_acc_obj.mtx[chunk_pos : chunk_pos_right])
    # and the mtx output name
    mtx_chunk_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_', str(chunk_pos), '_', str(chunk_pos_right), '.mtx'])
    # write the values
    feature_names_chunk.to_csv(feature_names_chunk_loc, sep = '\t', header = False, index = False, compression = 'gzip')
    io.mmwrite(mtx_chunk_loc, sparse.csr_matrix(mtx_chunk))
    # increase chunk position
    chunk_pos = chunk_pos + feature_chunk_size

In [9]:
# export the topic contribution matrix as well
topic_contribution_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/differential_accessibility/topic_annotations/', 'topic_contributions.tsv.gz'])
cistopic_object.selected_model.cell_topic.to_csv(topic_contribution_loc, sep = '\t', header = True, index = True, compression = 'gzip')